#### Analyze crawler output for RA

In [ ]:
from pathlib import Path
import pandas as pd


import monai
from monai.data import Dataset, CacheDataset, DataLoader

import monai
from monai.data import PILReader
from monai.transforms import LoadImage, LoadImaged, Resized, Compose, SaveImage, Spacingd, SpatialCropd, ResizeWithPadOrCropd


import numpy as np

In [ ]:

def extract_extras_from_filename(filename: str): 
    x = filename.split(".dcm")[0].split("_")
    d = {
        "filename": filename.replace(".dcm",""), 
        "id": x[0],
        "date_str": x[1],
        "sex": x[2],
        "left_or_right": x[3],
        # I dont know what the rest means:  dp_MTwo_InvNo_RotNo_BOk_OPNo_app_ComNo
        "x4": x[4],
        "x5": x[5],
        "x6": x[6],
        "x7": x[7],
        "x8": x[8],     
        "x9": x[9],    
        "x10": x[10],    
        "x11": x[11]                                        
    }
    return d

def extract_extras_from_abspath(abs_path):
    filename = str(abs_path).replace("._","").split(str("/"))[-1]
    return {**{"image": abs_path, **extract_extras_from_filename(filename)}}

import pydicom
import matplotlib.pyplot as plt

# Load the DICOM file
def plot_landmarks_from_df(dfm, image_idx=0):
    dcm_path = dfm["image"].iloc[image_idx]
    dcm = pydicom.dcmread(dcm_path)

    # Extract image data
    image = dcm.pixel_array

    # Create a larger figure
    plt.figure(figsize=(12, 10))  # Adjust the size as needed

    # Display the image
    plt.imshow(image, cmap='gray')

    # Landmark index
    for lm_idx in range(dfm.filter(regex="^landmark").shape[1]//2):
        point_x = dfm[f"landmark_{lm_idx}_0"].iloc[image_idx]  # X coordinate
        point_y = dfm[f"landmark_{lm_idx}_1"].iloc[image_idx]  # Y coordinate
        # plt.scatter(point_x, point_y, s=20, label=f"{lm_idx}") 
        plt.scatter(point_x, point_y, s=20, label=f"{lm_idx}", color='red', marker='x') 

    plt.axis("off")  # Hide axes
    #plt.title("DICOM Image with Landmarks")
    #plt.legend()
    plt.show()

In [ ]:


# file = "/home/cwatzenboeck/data/AutoPIX_local_data/tabular/metadata_crawler/Arthritis_crawler.csv"

# file = "/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/autoscoRA_data/autoscoRA_feet.csv"
# df = pd.read_csv(file)

#base_dir = Path("/project/autoscora/")
base_dir = Path("/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora")
folder_H = base_dir / "autoscoRA_images/H_images_of_interest_2_renamed_mirrored_inverted_dicoms"
folder_F = base_dir / "autoscoRA_images/F_images_of_interest_2_renamed_mirrored_inverted_dicoms"

# labels for landmark detection: 
df_lm_labels_H = base_dir / "landmark_data/100_all_H_joints36/points.csv"
df_lm_labels_F = base_dir / "landmark_data/100_all_F_joints27/points.csv"



In [ ]:
#df = pd.read_csv(df_lm_labels_F, header=None)

df = pd.read_csv(df_lm_labels_H, header=None)
column_names = ["filename"] + [f"landmark_{(i-1) // 2}_{(i-1) % 2}" for i in range(1, df.shape[1])]
df.columns = column_names
# drop those where on data is available
m = ~(df.filter(regex="^landmark")==0).all(axis=1)
df = df[m]

In [ ]:
files_F = list(folder_F.glob("*.dcm"))
files_H = list(folder_H.glob("*.dcm"))


#files_F_with_extras = [{"image": str(extract_extras_from_abspath(file)["image"])} for file in files_F]
files_F_with_extras = [extract_extras_from_abspath(file) for file in files_F[:]]
files_H_with_extras = [extract_extras_from_abspath(file) for file in files_H[:]]
df_images = pd.DataFrame(files_H_with_extras)


# TODO merge
dfm = pd.merge(df_images, df, on="filename", how="inner")
dfm

In [ ]:

plot_landmarks_from_df(dfm)

In [ ]:
dfm.filter(regex="^landmark").shape[1]

In [ ]:
# I want to load a dcm image (CR 2d) with monai. Improve the code below 

files_F_with_extras = [{"image": str(extract_extras_from_abspath(file)["image"])} for file in files_F]

transform = Compose([
    LoadImaged(keys=["image"], ensure_channel_first=True, reader="PydicomReader"),
    #Resized(keys=["image"], spatial_size=(512,512))
    #Spacingd(keys=["image"], pixdim=(, 1.0e-2), mode="bilinear"),
    #ResizeWithPadOrCropd(keys=["image"], spatial_size=(128, 128))  # Adjust spatial_size as required
])


dataset = Dataset(files_F_with_extras, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, num_workers=2)

X = next(iter(dataloader))


# TODO plot X["image"]

In [ ]:
import matplotlib.pyplot as plt
import torch

# Assuming X["image"] is a torch.Tensor of shape (B, C, H, W)
images = X["image"]

# Convert tensor to numpy array if needed
if isinstance(images, torch.Tensor):
    images = images.numpy()

# Loop over each image in the batch and plot
for i in range(images.shape[0]):
    img = images[i]  # shape: (C, H, W)
    # If the image has only one channel, remove the channel dimension for plotting
    if img.shape[0] == 1:
        img = img.squeeze(0)
    plt.figure()
    plt.imshow(img, cmap="gray")
    plt.title(f"Image {i}")
    plt.axis("off")

plt.show()
